In [19]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from scipy.stats import zscore

In [31]:
# 1. Load the datasets
train_data = pd.read_csv('../data/train.csv')
test_data = pd.read_csv('../data/test.csv')

# 2. Print information about train.csv
print("Number of rows in train.csv:", len(train_data))
print("Column names in train.csv:", train_data.columns.tolist())

# 3. Display the first 5 rows of the training dataset
print("\nFirst 5 rows of train.csv:")
print(train_data.head())

Number of rows in train.csv: 1317
Column names in train.csv: ['Make', 'Model', 'Price', 'Year', 'Kilometer', 'Fuel Type', 'Transmission', 'Location', 'Color', 'Owner', 'Seller Type', 'Engine', 'Max Power', 'Max Torque', 'Drivetrain', 'Length', 'Width', 'Height', 'Seating Capacity', 'Fuel Tank Capacity']

First 5 rows of train.csv:
        Make                                    Model    Price  Year  \
0  Ssangyong                               Rexton RX7   975000  2013   
1    Hyundai  Creta SX (O) 1.5 Petrol CVT [2020-2022]  1748999  2022   
2       Audi                     A4 2.0 TDI (143 bhp)  1150000  2012   
3    Hyundai        Grand i10 Magna AT 1.2 Kappa VTVT   549000  2018   
4    Hyundai                       Elite i20 Asta 1.2   675000  2017   

   Kilometer Fuel Type Transmission   Location   Color   Owner Seller Type  \
0      72000    Diesel    Automatic  Bangalore  Silver   First  Individual   
1       2670    Petrol    Automatic    Kolkata   White   First  Individual   


In [36]:
# Helper function to extract numeric value and RPM
def extract_value_and_rpm(val):
    """Extract numeric value and RPM from string (e.g., '123 bhp @ 4000 rpm')."""
    try:
        parts = re.findall(r"(\d+\.?\d*)", str(val))
        if len(parts) >= 2:
            return float(parts[0]), float(parts[1])
        elif len(parts) == 1:
            return float(parts[0]), np.nan
        else:
            return np.nan, np.nan
    except:
        return np.nan, np.nan

# Helper function to process Engine column
def process_engine_column(series):
    """Process Engine column to remove 'cc' and convert to float, handling non-string and NaN values."""
    def convert_engine(val):
        if pd.isna(val):
            return np.nan
        if isinstance(val, (int, float)):
            return float(val)
        if isinstance(val, str):
            return float(val.replace('cc', '').strip())
        return np.nan
    return series.apply(convert_engine)

# Preprocess train data
train_data[['Max_Power_Value', 'Max_Power_RPM']] = train_data['Max Power'].apply(lambda x: pd.Series(extract_value_and_rpm(x)))
train_data[['Max_Torque_Value', 'Max_Torque_RPM']] = train_data['Max Torque'].apply(lambda x: pd.Series(extract_value_and_rpm(x)))
train_data['Engine'] = process_engine_column(train_data['Engine'])

# Select features and target
selected_features = ['Max_Power_Value', 'Max_Torque_Value', 'Length', 'Fuel Tank Capacity']
# Check for target column
target_column = 'Price'
if 'Price' not in train_data.columns:
    possible_targets = ['price', 'PRICE', 'SalePrice', 'sale_price']
    for col in possible_targets:
        if col in train_data.columns:
            target_column = col
            print(f"Using '{col}' as target column instead of 'Price'")
            break
    else:
        raise KeyError("No target column ('Price' or variations) found in train.csv")

# Remove rows with missing values in selected features or target
mask = train_data[selected_features + [target_column]].notna().all(axis=1)
train_data = train_data[mask].reset_index(drop=True)

# Remove outliers based on Z-score for target
z_scores = zscore(train_data[target_column])
mask_outlier = np.abs(z_scores) < 3
train_data = train_data[mask_outlier].reset_index(drop=True)

# Select features and target
X_train = train_data[selected_features]
y_train = train_data[target_column]
y_train_log = np.log1p(y_train)  # Log transform for target

# Standardize numeric features
def standardize(X):
    """Standardize data to mean=0, std=1."""
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    return (X - mean) / std, mean, std

X_train_scaled, train_mean, train_std = standardize(X_train)

# Save preprocessed train data (include Price and Log Price)
train_data_preprocessed = X_train_scaled.copy()
train_data_preprocessed[target_column] = y_train
train_data_preprocessed['Log_' + target_column] = y_train_log
train_data_preprocessed.to_csv('../data/train_data_preprocessed_new.csv', index=False)

# Preprocess test data
test_data[['Max_Power_Value', 'Max_Power_RPM']] = test_data['Max Power'].apply(lambda x: pd.Series(extract_value_and_rpm(x)))
test_data[['Max_Torque_Value', 'Max_Torque_RPM']] = test_data['Max Torque'].apply(lambda x: pd.Series(extract_value_and_rpm(x)))
test_data['Engine'] = process_engine_column(test_data['Engine'])

# Check for target column in test data
if 'Price' not in test_data.columns:
    possible_targets = ['price', 'PRICE', 'SalePrice', 'sale_price']
    for col in possible_targets:
        if col in test_data.columns:
            target_column = col
            print(f"Using '{col}' as target column in test.csv")
            break
    else:
        raise KeyError("No target column ('Price' or variations) found in test.csv")

# Remove rows with missing values in selected features or target
mask = test_data[selected_features + [target_column]].notna().all(axis=1)
test_data = test_data[mask].reset_index(drop=True)

# Select features and target
X_test = test_data[selected_features]
y_test = test_data[target_column]
y_test_log = np.log1p(y_test)

# Standardize test data
X_test_scaled = (X_test - train_mean) / train_std

# Save preprocessed test data (include Price and Log Price)
test_data_preprocessed = X_test_scaled.copy()
test_data_preprocessed[target_column] = y_test
test_data_preprocessed['Log_' + target_column] = y_test_log
test_data_preprocessed.to_csv('../data/test_data_preprocessed_new.csv', index=False)

# Print first 5 rows of preprocessed train data
print("\nPreprocessed train data (first 5 rows):")
print(train_data_preprocessed.head())


Preprocessed train data (first 5 rows):
   Max_Power_Value  Max_Torque_Value    Length  Fuel Tank Capacity    Price  \
0         1.253796          1.415593  1.252478            1.910888   975000   
1        -0.124817         -0.664674  0.166328           -0.045852  1748999   
2        -0.746163         -0.906566 -1.110794           -0.535037   549000   
3        -0.726746         -0.898503 -0.585622           -0.395269   675000   
4        -0.590826         -0.180891  0.144843           -0.535037   295000   

   Log_Price  
0  13.790194  
1  14.374555  
2  13.215856  
3  13.422469  
4  12.594734  


In [38]:
# LinearRegression class
class LinearRegression:
    def __init__(self, alpha=0.01, l2_lambda=0.1, num_iterations=10000, transform_func=None):
        self.alpha = alpha
        self.l2_lambda = l2_lambda
        self.num_iterations = num_iterations
        self.transform_func = transform_func
        self.weights = None
        self.bias = 0

    def fit(self, X, y):
        """Train model using gradient descent."""
        X_trans = self.transform_func(X) if self.transform_func else X
        if self.weights is None:
            self.weights = np.zeros(X_trans.shape[1])
        for _ in range(self.num_iterations):
            predictions = np.dot(X_trans, self.weights) + self.bias
            errors = predictions - y
            gradient_w = (X_trans.T.dot(errors) + 2 * self.l2_lambda * self.weights) / len(y)
            gradient_b = np.mean(errors)
            self.weights -= self.alpha * gradient_w
            self.bias -= self.alpha * gradient_b

    def predict(self, X):
        """Predict using transformed features."""
        X_trans = self.transform_func(X) if self.transform_func else X
        return np.dot(X_trans, self.weights) + self.bias
    
    def get_weights(self):
        """Return weights and bias."""
        return self.weights, self.bias

# Define transformation functions for the four equations
def make_transform_func_1():
    """Equation 1: y = a1*x1 + a2*x2 + a3*x3 + a4*x4"""
    def transform_func(X):
        return X
    return transform_func

def make_transform_func_2():
    """Equation 2: y = a1*x1^2 + a2*x2 + a3*x3^2 + a4*x4"""
    def transform_func(X):
        return np.hstack([X[:, 0:1]**2, X[:, 1:2], X[:, 2:3]**2, X[:, 3:4]])
    return transform_func

def make_transform_func_3():
    """Equation 3: y = a1*(x1 + x2) + a3*x3^2 + a4*x4"""
    def transform_func(X):
        return np.hstack([(X[:, 0:1] + X[:, 1:2]), X[:, 2:3]**2, X[:, 3:4]])
    return transform_func

def make_transform_func_4():
    """Equation 4: y = a1*x1*x2 + a3*x3^2"""
    def transform_func(X):
        return np.hstack([(X[:, 0:1] * X[:, 1:2]), X[:, 2:3]**2])
    return transform_func

# Evaluation function with MSE, MAE, RMSE, and R2
def evaluate_regression(model, X, y, y_original=None, metrics=['mse', 'mae', 'rmse', 'r2'], use_log=True):
    """Evaluate model with specified metrics."""
    y_pred_log = model.predict(X)
    if use_log and y_original is not None:
        y_pred = np.expm1(y_pred_log)  # Convert from log to original scale
        y_true = y_original  # Use original y for metrics
    else:
        y_pred = y_pred_log
        y_true = y
    
    results = {}
    if 'mse' in metrics:
        mse = np.mean((y_pred - y_true) ** 2)
        results['mse'] = mse
    if 'mae' in metrics:
        mae = np.mean(np.abs(y_pred - y_true))
        results['mae'] = mae
    if 'rmse' in metrics:
        rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
        results['rmse'] = rmse
    if 'r2' in metrics:
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
        results['r2'] = r2
    
    return results

# Train and evaluate four models
transform_funcs = [
    make_transform_func_1(),
    make_transform_func_2(),
    make_transform_func_3(),
    make_transform_func_4()
]

X_train_np = X_train_scaled.to_numpy()
y_train_np = y_train_log.to_numpy()  # Use log-transformed target
X_test_np = X_test_scaled.to_numpy()
y_test_np = y_test_log.to_numpy()  # Use log-transformed target

results = {}
for i, transform_func in enumerate(transform_funcs, 1):
    # Initialize and train model
    model = LinearRegression(transform_func=transform_func, alpha=0.01, num_iterations=10000)
    model.fit(X_train_np, y_train_np)
    
    # Evaluate on training set
    eval_results = evaluate_regression(
        model, 
        X_train_np, 
        y_train_np, 
        y_original=y_train.to_numpy(),  # Original Price for metrics
        metrics=['mse', 'mae', 'rmse', 'r2'],
        use_log=True
    )
    
    # Store results
    weights, bias = model.get_weights()
    results[i] = {'weights': weights, 'bias': bias, **eval_results}
    
    # Print results
    print(f"\nEquation {i}:")
    print(f"Weights: {weights}")
    print(f"Bias: {bias}")
    print(f"Training MSE: {eval_results['mse']}")
    print(f"Training MAE: {eval_results['mae']}")
    print(f"Training RMSE: {eval_results['rmse']}")
    print(f"Training R2: {eval_results['r2']}")

# Function to evaluate model on a new CSV file
def evaluate_model_on_file(file_path, model, metric='mse'):
    """Evaluate model on a new CSV file."""
    test_data = pd.read_csv(file_path)
    # Check for target column
    target_col = 'Price'
    if 'Price' not in test_data.columns:
        possible_targets = ['price', 'PRICE', 'SalePrice', 'sale_price']
        for col in possible_targets:
            if col in test_data.columns:
                target_col = col
                print(f"Using '{col}' as target column in {file_path}")
                break
        else:
            raise KeyError(f"No target column ('Price' or variations) found in {file_path}")
    
    # Check for log-transformed target
    log_target_col = 'Log_' + target_col
    if log_target_col not in test_data.columns:
        raise KeyError(f"Log-transformed target column '{log_target_col}' not found in {file_path}")
    
    X_test = test_data[selected_features].to_numpy()
    y_test_log = test_data[log_target_col].to_numpy()
    y_test = test_data[target_col].to_numpy()
    
    # Evaluate
    results = evaluate_regression(
        model, 
        X_test, 
        y_test_log, 
        y_original=y_test, 
        metrics=[metric], 
        use_log=True
    )
    print(f"{metric.upper()} on {file_path}: {results[metric]}")
    return results[metric]

# Example: Evaluate the first model on test data
best_model = LinearRegression(transform_func=transform_funcs[0], alpha=0.01, num_iterations=10000)
best_model.fit(X_train_np, y_train_np)

# Evaluate on test data with all metrics
test_results = evaluate_regression(
    best_model, 
    X_test_np, 
    y_test_np, 
    y_original=y_test.to_numpy(), 
    metrics=['mse', 'mae', 'rmse', 'r2'], 
    use_log=True
)
print("\nTest set results:")
print(f"Test MSE: {test_results['mse']}")
print(f"Test MAE: {test_results['mae']}")
print(f"Test RMSE: {test_results['rmse']}")
print(f"Test R2: {test_results['r2']}")

# Evaluate on file
evaluate_model_on_file('../data/test_data_preprocessed_new.csv', best_model, metric='mse')


Equation 1:
Weights: [ 0.28026206  0.2864623   0.17273205 -0.0447087 ]
Bias: 13.65299222566006
Training MSE: 991550258010.266
Training MAE: 495629.92223477736
Training RMSE: 995766.1663313662
Training R2: 0.15669075703404334

Equation 2:
Weights: [-0.00419379  0.57654953 -0.09987465  0.09061993]
Bias: 13.757060663592418
Training MSE: 646266560887.7534
Training MAE: 485768.89579082106
Training RMSE: 803907.0598568926
Training R2: 0.4503530609632286

Equation 3:
Weights: [ 0.33849201 -0.11105232  0.01983927]
Bias: 13.764044546668622
Training MSE: 1632703616551.3557
Training MAE: 500002.1714665102
Training RMSE: 1277772.9127475491
Training R2: -0.3886074253307774

Equation 4:
Weights: [ 0.23134512 -0.19953415]
Bias: 13.648208867905788
Training MSE: 121453201146882.47
Training MAE: 1009504.6766952397
Training RMSE: 11020580.798981626
Training R2: -102.29542682032067

Test set results:
Test MSE: 2601609425226.673
Test MAE: 617166.7201349975
Test RMSE: 1612950.5340296933
Test R2: 0.16735500

2601609425226.673

# Lý do chọn các phương trình hồi quy

1. **Phương trình 1: y = a1*x1 + a2*x2 + a3*x3 + a4*x4**
   - **Lý do**: Mô hình tuyến tính cơ bản, giả định mối quan hệ tuyến tính giữa các đặc trưng (`Max_Power_Value`, `Max_Torque_Value`, `Length`, `Fuel Tank Capacity`) và giá xe. Đây là baseline để so sánh với các mô hình phức tạp hơn.

2. **Phương trình 2: y = a1*x1^2 + a2*x2 + a3*x3^2 + a4*x4**
   - **Lý do**: Bình phương `Max_Power_Value` và `Length` để mô hình hóa ảnh hưởng phi tuyến, vì công suất động cơ và chiều dài xe có thể ảnh hưởng không tuyến tính đến giá (ví dụ: xe dài hơn hoặc mạnh hơn có giá tăng không đều).

3. __Phương trình 3: y = a1*(x1 + x2) + a3*x3^2 + a4*x4__
   - **Lý do**: Kết hợp `Max_Power_Value` và `Max_Torque_Value` vì chúng có tương quan cao với giá (0.767 và 0.737 từ `dp.ipynb`). Bình phương `Length` để kiểm tra ảnh hưởng phi tuyến của kích thước xe.

4. **Phương trình 4: y = a1*x1*x2 + a3*x3^2**
   - **Lý do**: Tương tác giữa `Max_Power_Value` và `Max_Torque_Value` mô hình hóa hiệu ứng kết hợp của hai đặc trưng quan trọng. Bình phương `Length` giữ yếu tố phi tuyến, giảm số tham số để tập trung vào các đặc trưng chính.

**Quan sát**: Các đặc trưng được chọn dựa trên tương quan cao với giá xe (theo `feature_scores` trong `dp.ipynb`). Phương trình 1 đơn giản và dễ diễn giải. Phương trình 2 và 3 kiểm tra các hiệu ứng phi tuyến và kết hợp, trong khi phương trình 4 tập trung vào tương tác, phù hợp với các đặc trưng có ảnh hưởng mạnh.